In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")

# Parse timestamps to UTC datetimes as required (keep original integer timestamps too)
df_metric["timestamp_dt"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
df_trace["timestamp_dt"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)
df_log["timestamp_dt"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)

# Aggregation helper function to keep consistent column names
def aggregate_series(df, group_cols, value_col_name):
    agg = df.groupby(group_cols, as_index=False).agg(
        count=(value_col_name, "count"),
        min_timestamp=("timestamp", "min"),
        max_timestamp=("timestamp", "max"),
        mean_value=(value_col_name, "mean"),
        median_value=(value_col_name, "median"),
        p95_value=(value_col_name, lambda x: x.quantile(0.95)),
        max_value=(value_col_name, "max"),
    )
    # sort by count desc and reset index
    agg = agg.sort_values("count", ascending=False).reset_index(drop=True)
    return agg

# 1) metric.csv aggregation
metric_agg = aggregate_series(df_metric, ["cmdb_id", "kpi_name"], "value")
metric_top50 = metric_agg.head(50)
metric_cmdb_ids = pd.Series(np.sort(df_metric["cmdb_id"].unique()), name="metric_cmdb_id")

# 2) trace.csv aggregation
trace_agg = aggregate_series(df_trace, ["cmdb_id", "trace_name"], "value")
trace_top50 = trace_agg.head(50)
trace_cmdb_ids = pd.Series(np.sort(df_trace["cmdb_id"].unique()), name="trace_cmdb_id")

# 3) log.csv aggregation
log_agg = aggregate_series(df_log, ["cmdb_id", "log_name"], "value")
log_top50 = log_agg.head(50)
log_cmdb_ids = pd.Series(np.sort(df_log["cmdb_id"].unique()), name="log_cmdb_id")

# Display compact aggregated tables and unique cmdb_id lists (variables shown, not prints)
metric_top50, metric_cmdb_ids, trace_top50, trace_cmdb_ids, log_top50, log_cmdb_ids

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregates:\n"
    "- metric.csv unique cmdb_id values: adservice, cartservice, checkoutservice, currencyservice, "
    "emailservice, frontend, frontend-external, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice.\n"
    "- trace.csv unique cmdb_id values: checkoutservice, currencyservice, emailservice, frontendservice, paymentservice, productcatalogservice, recommendationservice, root.\n"
    "- log.csv unique cmdb_id values: adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, redis, shippingservice.\n\n"
    "Key findings from aggregated groups (top observations):\n"
    "- Most metric/trace/log groups have 25 samples (consistent coverage); redis logs have only 5 samples.\n"
    "- log.error_count is 0.0 for all services shown (no aggregated log error counts in these groups).\n"
    "- High resource metric values observed:\n"
    "  • adservice: very high diskio (mean ~6.76e4, max ~1.37e5) and large memory values (mean ~1.14e8).\n"
    "  • paymentservice: notable memory (mean ~4.07e7).\n"
    "  • productcatalogservice: higher CPU (mean ~3.39, max ~3.63) relative to others.\n"
    "- Trace highlights:\n"
    "  • checkoutservice traces are prominent (high counts); row_count mean ~212.8 (max 316) and frontend→checkout duration_mean ~0.32s (max ~0.44s).\n"
    "  • trace error_rate groups show 0.0 aggregated error rates in the displayed results.\n\n"
    "Overall: telemetry coverage is good across services (mostly 25 samples). No aggregated log or trace error rates are present in the summaries, but resource metrics indicate potential hotspots (especially adservice disk I/O and memory, and notable memory on paymentservice) that warrant further, time-windowed investigation if investigating performance issues."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count  min_timestamp  max_timestamp    mean_value  median_value     p95_value     max_value
0               adservice         cpu     25     1705365780     1705367220  6.187571e-01  6.111566e-01  6.784633e-01  7.331173e-01
1               adservice      diskio     25     1705365780     1705367220  6.760010e+04  6.243024e+04  1.090259e+05  1.375374e+05
2               adservice  latency-50     25     1705365780     1705367220  2.972444e-03  2.971448e-03  2.988632e-03  2.998487e-03
3               adservice  latency-90     25     1705365780     1705367220  4.604855e-03  4.600342e-03  4.619029e-03  4.638788e-03
4               adservice         mem     25     1705365780     1705367220  1.138074e+08  1.138149e+08  1.139134e+08  1.139296e+08
..                    ...         ...    ...            ...            ...           ...           ...           ...           ...
45         paymentservice         mem     25     1705365780     1705367220  4.065929e+07  4.056774e+07  4.116606e+07  4.117217e+07
46         paymentservice      socket     25     1705365780     1705367220  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
47         paymentservice    workload     25     1705365780     1705367220  8.761233e-01  9.122167e-01  1.029421e+00  1.088883e+00
48  productcatalogservice         cpu     25     1705365780     1705367220  3.387204e+00  3.431330e+00  3.559068e+00  3.630422e+00
49  productcatalogservice      diskio     25     1705365780     1705367220  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00

[50 rows x 9 columns], 0                 adservice
1               cartservice
2           checkoutservice
3           currencyservice
4              emailservice
5                  frontend
6         frontend-external
7            paymentservice
8     productcatalogservice
9     recommendationservice
10                    redis
11          shippingservice
Name: metric_cmdb_id, dtype: object,             cmdb_id                                trace_name  count  min_timestamp  max_timestamp  mean_value  median_value   p95_value   max_value
0   checkoutservice  trace.from_checkoutservice.duration_mean     25     1705365780     1705367220    0.004917      0.003732    0.008454    0.008546
1   checkoutservice   trace.from_checkoutservice.duration_p95     25     1705365780     1705367220    0.005736      0.005802    0.006207    0.006651
2   checkoutservice     trace.from_checkoutservice.error_rate     25     1705365780     1705367220    0.000000      0.000000    0.000000    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25     1705365780     1705367220  212.760000    221.000000  259.400000  316.000000
4   checkoutservice  trace.from_frontendservice.duration_mean     25     1705365780     1705367220    0.319686      0.324897    0.405374    0.437474
..              ...                                       ...    ...            ...            ...         ...           ...         ...         ...
45     emailservice   trace.from_checkoutservice.duration_p95     25     1705365780     1705367220    0.000513      0.000471    0.000766    0.000831
46     emailservice     trace.from_checkoutservice.error_rate     25     1705365780     1705367220    0.000000      0.000000    0.000000    0.000000
47     emailservice      trace.from_checkoutservice.row_count     25     1705365780     1705367220   25.240000     27.000000   31.600000   36.000000
48     emailservice             trace.from_root.duration_mean     25     1705365780     1705367220    0.000045      0.000046    0.000052    0.000055
49     emailservice              trace.from_root.duration_p95     25     1705365780     1705367220    0.000071      0.000067    0.000111    0.000126

[50 rows x 9 columns], 0          checkoutservice
1          currencyservice
2             emailservice
3          frontendservice
4           paymentservice
5    productcatalogservice
6    recommendationservice
7                     root
Name: trace_cmdb_id, dtype: object,                   cmdb_id         log_name  count  min_timestamp  max_timestamp  mean_value  median_value  p95_value  max_value
0               adservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
1               adservice    log.row_count     25     1705365780     1705367220      415.92         435.0      448.0        456
2             cartservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
3             cartservice    log.row_count     25     1705365780     1705367220      708.40         735.0      774.6        779
4         checkoutservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
5         checkoutservice    log.row_count     25     1705365780     1705367220       75.72          82.0       95.0        108
6         currencyservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
7         currencyservice    log.row_count     25     1705365780     1705367220     2162.48        2211.0     2372.8       2427
8            emailservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
9            emailservice    log.row_count     25     1705365780     1705367220       25.24          27.0       31.6         36
10               frontend  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
11               frontend    log.row_count     25     1705365780     1705367220     2437.20        2532.0     2664.4       2686
12         paymentservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
13         paymentservice    log.row_count     25     1705365780     1705367220       50.48          54.0       63.2         72
14  recommendationservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
15  recommendationservice    log.row_count     25     1705365780     1705367220      521.00         541.0      564.6        574
16        shippingservice    log.row_count     25     1705365780     1705367220      408.56         422.0      478.0        520
17        shippingservice  log.error_count     25     1705365780     1705367220        0.00           0.0        0.0          0
18                  redis    log.row_count      5     1705365840     1705367040        5.00           5.0        5.0          5
19                  redis  log.error_count      5     1705365840     1705367040        0.00           0.0        0.0          0, 0                adservice
1              cartservice
2          checkoutservice
3          currencyservice
4             emailservice
5                 frontend
6           paymentservice
7    recommendationservice
8                    redis
9          shippingservice
Name: log_cmdb_id, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse previously loaded df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp_dt exists (per timezone rule) but keep integer timestamps for outputs
if "timestamp_dt" not in df_metric.columns:
    df_metric["timestamp_dt"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# 1) Compute global P95 per (cmdb_id, kpi_name) using full available data (no filtering)
global_p95_df = (
    df_metric.groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(global_p95=("value", lambda x: float(x.quantile(0.95))))
)

# Keep only candidate services' groups
global_p95_candidates = global_p95_df[global_p95_df["cmdb_id"].isin(candidates)].copy()

# 2) Define incident window (UTC) and filter metric.csv to that window
start_ts = int(pd.Timestamp("2024-01-16 00:40:22", tz="UTC").timestamp())
end_ts = int(pd.Timestamp("2024-01-16 01:10:22", tz="UTC").timestamp())

df_window = df_metric[
    (df_metric["cmdb_id"].isin(candidates))
    & (df_metric["timestamp"] >= start_ts)
    & (df_metric["timestamp"] <= end_ts)
].copy()

# Merge global P95 into window rows so we can evaluate exceedances per-row
df_window = df_window.merge(global_p95_candidates, on=["cmdb_id", "kpi_name"], how="left")

# Flag exceedances (strictly greater than global P95)
df_window["exceed"] = df_window["value"] > df_window["global_p95"]

# Group windowed data to compute required in-window stats
window_agg = (
    df_window.groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        anomalies_count_in_window=("exceed", "sum"),
        earliest_anomaly_timestamp=("timestamp", lambda s: int(s[df_window.loc[s.index, "exceed"]].min()) if any(df_window.loc[s.index, "exceed"]) else np.nan),
        max_value_in_window=("value", "max"),
    )
)

# Note: the lambda above uses index alignment to find earliest exceeding timestamp per group.
# However, groups with no rows in window won't appear in window_agg. We'll left-join to global list.

# 3) Assemble final table: for each candidate group include global_p95 and window stats (fill zeros/nans where appropriate)
final = global_p95_candidates.merge(window_agg, on=["cmdb_id", "kpi_name"], how="left")

# Fill anomalies_count_in_window NaN -> 0 (no rows in window means zero anomalies)
final["anomalies_count_in_window"] = final["anomalies_count_in_window"].fillna(0).astype(int)

# earliest_anomaly_timestamp remains NaN where no anomaly; ensure integer where present
final["earliest_anomaly_timestamp"] = final["earliest_anomaly_timestamp"].where(final["earliest_anomaly_timestamp"].notna(), np.nan)
# max_value_in_window stays NaN if no rows in window; otherwise keep numeric
final["max_value_in_window"] = final["max_value_in_window"].astype(float)

# Select and order columns as requested
final = final[
    ["cmdb_id", "kpi_name", "global_p95", "anomalies_count_in_window", "earliest_anomaly_timestamp", "max_value_in_window"]
]

# Sort by anomalies_count_in_window descending, then cmdb_id
final = final.sort_values(["anomalies_count_in_window", "cmdb_id"], ascending=[False, True]).reset_index(drop=True)

# Limit to top 50 rows (dataset expected to be small)
final_top50 = final.head(50)

# Display the compact aggregated table
final_top50

```
Out[2]:
```
```python
# Summarize anomalies from the previously computed `final` table for the incident window.
# Reuse variables from the notebook state (final, candidates).
try:
    final
except NameError:
    import pandas as pd
    final = pd.read_csv("metric.csv")  # fallback (should not be needed)

candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

lines = []
lines.append("Incident window: 2024-01-16 00:40:22 to 2024-01-16 01:10:22 UTC")
lines.append("Summary of metric anomalies (value > global P95). Timestamps in integer seconds.\n")

for svc in candidates:
    svc_rows = final[final["cmdb_id"] == svc].sort_values("anomalies_count_in_window", ascending=False)
    total_anoms = int(svc_rows["anomalies_count_in_window"].sum()) if not svc_rows.empty else 0
    lines.append(f"{svc}:")
    if svc_rows.empty or total_anoms == 0:
        lines.append("  - No anomalies detected (no values exceeding global P95 in the window).")
    else:
        lines.append(f"  - Total KPI groups with anomalies: {int((svc_rows['anomalies_count_in_window']>0).sum())}, total anomaly points: {total_anoms}")
        # List KPIs with anomalies (limit to those with anomalies)
        for _, r in svc_rows[svc_rows["anomalies_count_in_window"] > 0].iterrows():
            k = r["kpi_name"]
            cnt = int(r["anomalies_count_in_window"])
            et = int(r["earliest_anomaly_timestamp"]) if pd.notna(r["earliest_anomaly_timestamp"]) else "N/A"
            maxv = r["max_value_in_window"]
            p95 = r["global_p95"]
            lines.append(
                f"    - {k}: anomalies={cnt}, earliest_ts={et}, max_in_window={maxv:.6g}, global_p95={p95:.6g}"
            )
    lines.append("")  # blank line between services

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  anomalies_count_in_window  earliest_anomaly_timestamp  max_value_in_window
0         checkoutservice         cpu  4.656845e-01                          2                1.705366e+09         5.258337e-01
1         checkoutservice  latency-50  3.370723e-01                          2                1.705366e+09         4.046768e-01
2         checkoutservice  latency-90  8.816422e-01                          2                1.705366e+09         1.272944e+00
3         checkoutservice         mem  1.221923e+07                          2                1.705366e+09         1.241675e+07
4         checkoutservice    workload  1.024707e+00                          2                1.705366e+09         1.088917e+00
..                    ...         ...           ...                        ...                         ...                  ...
30           emailservice      socket  3.000000e+00                          0                         NaN         3.000000e+00
31  productcatalogservice      diskio  0.000000e+00                          0                         NaN         0.000000e+00
32  productcatalogservice       error  0.000000e+00                          0                         NaN         0.000000e+00
33  productcatalogservice      socket  4.000000e+00                          0                         NaN         4.000000e+00
34  recommendationservice      socket  6.000000e+00                          0                         NaN         6.000000e+00

[35 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp_dt exists per timezone rule (UTC) but keep integer timestamps for outputs
if "timestamp_dt" not in df_trace.columns:
    df_trace["timestamp_dt"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# 1) Compute global P95 per (cmdb_id, trace_name) using full trace data (no filtering)
global_p95_trace = (
    df_trace.groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(global_p95=("value", lambda x: float(x.quantile(0.95))))
)

# Keep only candidate services' groups
global_p95_candidates = global_p95_trace[global_p95_trace["cmdb_id"].isin(candidates)].copy()

# 2) Define incident window (UTC) and filter trace.csv to that window
start_ts = int(pd.Timestamp("2024-01-16 00:40:22", tz="UTC").timestamp())
end_ts = int(pd.Timestamp("2024-01-16 01:10:22", tz="UTC").timestamp())

df_trace_window = df_trace[
    (df_trace["cmdb_id"].isin(candidates))
    & (df_trace["timestamp"] >= start_ts)
    & (df_trace["timestamp"] <= end_ts)
].copy()

# Merge global P95 into window rows so we can evaluate exceedances per-row
df_trace_window = df_trace_window.merge(global_p95_candidates, on=["cmdb_id", "trace_name"], how="left")

# Flag exceedances (strictly greater than global P95)
df_trace_window["exceed"] = df_trace_window["value"] > df_trace_window["global_p95"]

# Aggregation for exceedances: count and earliest timestamp (from exceed rows)
df_exceed = df_trace_window[df_trace_window["exceed"]].copy()
exceed_agg = (
    df_exceed.groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(
        anomalies_count_in_window=("value", "count"),
        earliest_anomaly_timestamp=("timestamp", "min"),
    )
)

# Aggregation for max value in window (including non-exceed rows)
max_agg = (
    df_trace_window.groupby(["cmdb_id", "trace_name"], as_index=False)
    .agg(max_value_in_window=("value", "max"))
)

# 3) Assemble final table: include groups from global_p95_candidates even if no rows in window
final = (
    global_p95_candidates
    .merge(exceed_agg, on=["cmdb_id", "trace_name"], how="left")
    .merge(max_agg, on=["cmdb_id", "trace_name"], how="left")
)

# Fill anomalies_count_in_window NaN -> 0 (no exceed rows)
final["anomalies_count_in_window"] = final["anomalies_count_in_window"].fillna(0).astype(int)

# earliest_anomaly_timestamp remain NaN where no exceed; convert to integer where present
final["earliest_anomaly_timestamp"] = final["earliest_anomaly_timestamp"].where(final["earliest_anomaly_timestamp"].notna(), np.nan)

# max_value_in_window stays NaN if no rows in window; otherwise numeric
final["max_value_in_window"] = final["max_value_in_window"].astype(float)

# Select and order columns as requested
final = final[["cmdb_id", "trace_name", "global_p95", "anomalies_count_in_window", "earliest_anomaly_timestamp", "max_value_in_window"]]

# Sort by anomalies_count_in_window descending then cmdb_id
final = final.sort_values(["anomalies_count_in_window", "cmdb_id"], ascending=[False, True]).reset_index(drop=True)

# Limit to top 20 rows
final_top20 = final.head(20)

# Display compact aggregated table
final_top20

```
Out[3]:
```
```python
# Summarize trace anomalies for the five candidate services using previously computed `final` table.
# The `final` DataFrame should contain columns: cmdb_id, trace_name, global_p95, anomalies_count_in_window, earliest_anomaly_timestamp, max_value_in_window.

try:
    final
except NameError:
    # fallback: recompute if not present (shouldn't be necessary in normal stateful kernel)
    import pandas as pd
    df_trace = pd.read_csv("trace.csv")
    df_trace["timestamp_dt"] = pd.to_datetime(df_trace["timestamp"], unit="s", utc=True)
    candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]
    global_p95_trace = df_trace.groupby(["cmdb_id","trace_name"], as_index=False).agg(global_p95=("value", lambda x: float(x.quantile(0.95))))
    start_ts = int(pd.Timestamp("2024-01-16 00:40:22", tz="UTC").timestamp())
    end_ts = int(pd.Timestamp("2024-01-16 01:10:22", tz="UTC").timestamp())
    df_trace_window = df_trace[(df_trace["cmdb_id"].isin(candidates)) & (df_trace["timestamp"] >= start_ts) & (df_trace["timestamp"] <= end_ts)].copy()
    df_trace_window = df_trace_window.merge(global_p95_trace[global_p95_trace["cmdb_id"].isin(candidates)], on=["cmdb_id","trace_name"], how="left")
    df_trace_window["exceed"] = df_trace_window["value"] > df_trace_window["global_p95"]
    exceed_agg = df_trace_window[df_trace_window["exceed"]].groupby(["cmdb_id","trace_name"], as_index=False).agg(anomalies_count_in_window=("value","count"), earliest_anomaly_timestamp=("timestamp","min"))
    max_agg = df_trace_window.groupby(["cmdb_id","trace_name"], as_index=False).agg(max_value_in_window=("value","max"))
    final = global_p95_trace[global_p95_trace["cmdb_id"].isin(candidates)].merge(exceed_agg, on=["cmdb_id","trace_name"], how="left").merge(max_agg, on=["cmdb_id","trace_name"], how="left")
    final["anomalies_count_in_window"] = final["anomalies_count_in_window"].fillna(0).astype(int)
    final["earliest_anomaly_timestamp"] = final["earliest_anomaly_timestamp"].where(final["earliest_anomaly_timestamp"].notna(), None)
    final["max_value_in_window"] = final["max_value_in_window"].astype(float)

candidates = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

lines = []
lines.append("Incident window: 2024-01-16 00:40:22 to 2024-01-16 01:10:22 UTC")
lines.append("Trace anomaly summary (value > global P95). Timestamps shown as integer seconds.\n")

for svc in candidates:
    svc_rows = final[final["cmdb_id"] == svc].sort_values("anomalies_count_in_window", ascending=False)
    total_anoms = int(svc_rows["anomalies_count_in_window"].sum()) if not svc_rows.empty else 0
    lines.append(f"{svc}:")
    if total_anoms == 0:
        lines.append("  - No trace metric values exceeded their global P95 in the window.")
    else:
        lines.append(f"  - Total anomaly points across trace metrics: {total_anoms}")
        # List trace metrics with anomalies
        for _, r in svc_rows[svc_rows["anomalies_count_in_window"] > 0].iterrows():
            tn = r["trace_name"]
            cnt = int(r["anomalies_count_in_window"])
            et = int(r["earliest_anomaly_timestamp"]) if r["earliest_anomaly_timestamp"] is not None and not pd.isna(r["earliest_anomaly_timestamp"]) else "NaN"
            maxv = r["max_value_in_window"]
            p95 = r["global_p95"]
            lines.append(f"    - {tn}: anomalies={cnt}, earliest_ts={et}, max_in_window={maxv:.6g}, global_p95={p95:.6g}")
    lines.append("")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name  global_p95  anomalies_count_in_window  earliest_anomaly_timestamp  max_value_in_window
0   checkoutservice  trace.from_checkoutservice.duration_mean    0.008454                          2                1.705366e+09             0.008546
1   checkoutservice   trace.from_checkoutservice.duration_p95    0.006207                          2                1.705366e+09             0.006651
2   checkoutservice      trace.from_checkoutservice.row_count  259.400000                          2                1.705366e+09           316.000000
3   checkoutservice  trace.from_frontendservice.duration_mean    0.405374                          2                1.705367e+09             0.437474
4   checkoutservice   trace.from_frontendservice.duration_p95    0.957077                          2                1.705367e+09             1.112333
5   checkoutservice      trace.from_frontendservice.row_count   31.800000                          2                1.705366e+09            36.000000
6   checkoutservice             trace.from_root.duration_mean    0.081081                          2                1.705367e+09             0.094318
7   checkoutservice              trace.from_root.duration_p95    0.194545                          2                1.705367e+09             0.277321
8   checkoutservice                 trace.from_root.row_count  133.600000                          2                1.705366e+09           160.000000
9   checkoutservice    trace.to_checkoutservice.duration_mean    0.008454                          2                1.705366e+09             0.008546
10  checkoutservice     trace.to_checkoutservice.duration_p95    0.006207                          2                1.705366e+09             0.006651
11  checkoutservice        trace.to_checkoutservice.row_count  259.400000                          2                1.705366e+09           316.000000
12  checkoutservice    trace.to_currencyservice.duration_mean    0.003061                          2                1.705367e+09             0.004422
13  checkoutservice     trace.to_currencyservice.duration_p95    0.000231                          2                1.705367e+09             0.030273
14  checkoutservice        trace.to_currencyservice.row_count  109.600000                          2                1.705366e+09           136.000000
15  checkoutservice       trace.to_emailservice.duration_mean    0.000348                          2                1.705367e+09             0.000400
16  checkoutservice        trace.to_emailservice.duration_p95    0.000766                          2                1.705367e+09             0.000831
17  checkoutservice           trace.to_emailservice.row_count   31.600000                          2                1.705366e+09            36.000000
18  checkoutservice     trace.to_paymentservice.duration_mean    0.000345                          2                1.705367e+09             0.000364
19  checkoutservice      trace.to_paymentservice.duration_p95    0.000876                          2                1.705366e+09             0.000920```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse df_metric from kernel state
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp_dt exists (UTC) but keep integer timestamps for outputs
if "timestamp_dt" not in df_metric.columns:
    df_metric["timestamp_dt"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Incident window (UTC)
start_ts = int(pd.Timestamp("2024-01-16 00:40:22", tz="UTC").timestamp())
end_ts = int(pd.Timestamp("2024-01-16 01:10:22", tz="UTC").timestamp())

# Focus on checkoutservice
checkout_df = df_metric[df_metric["cmdb_id"] == "checkoutservice"].copy()

# 1) Global P95 per KPI (full series for checkoutservice)
global_p95 = (
    checkout_df
    .groupby("kpi_name", as_index=False)
    .agg(global_p95=("value", lambda x: float(x.quantile(0.95))))
)

# 2) Filter checkoutservice rows to incident window
window_df = checkout_df[(checkout_df["timestamp"] >= start_ts) & (checkout_df["timestamp"] <= end_ts)].copy()

# 3) For each KPI compute required summaries
rows = []
for _, r in global_p95.iterrows():
    kpi = r["kpi_name"]
    gp95 = r["global_p95"]
    k_window = window_df[window_df["kpi_name"] == kpi].copy()
    
    if k_window.empty:
        anomalies_count = 0
        earliest_ts = np.nan
        max_in_window = np.nan
        longest_streak = 0
    else:
        # max value in window (even if no exceed)
        max_in_window = float(k_window["value"].max())
        # exceed rows strictly greater than global p95
        exceed = k_window[k_window["value"] > gp95].copy()
        anomalies_count = int(len(exceed))
        earliest_ts = int(exceed["timestamp"].min()) if anomalies_count > 0 else np.nan
        
        if anomalies_count > 0:
            # compute longest consecutive streak where successive timestamps differ by exactly 60 seconds
            ts_sorted = np.sort(exceed["timestamp"].unique().astype(int))
            if ts_sorted.size == 0:
                longest_streak = 0
            else:
                diffs = np.diff(ts_sorted)
                # boundaries where diff != 60
                # compute run lengths
                runs = []
                run_len = 1
                for d in diffs:
                    if d == 60:
                        run_len += 1
                    else:
                        runs.append(run_len)
                        run_len = 1
                runs.append(run_len)
                longest_streak = int(max(runs)) if runs else 1
        else:
            longest_streak = 0

    # breach_ratio: (max_in_window - gp95) / gp95, NaN if gp95 == 0 or max_in_window NaN
    if gp95 == 0 or np.isnan(max_in_window):
        breach_ratio = np.nan
    else:
        breach_ratio = (max_in_window - gp95) / gp95

    rows.append({
        "kpi_name": kpi,
        "global_p95": gp95,
        "anomalies_count_in_window": anomalies_count,
        "earliest_anomaly_timestamp": earliest_ts,
        "max_value_in_window": max_in_window,
        "breach_ratio": breach_ratio,
        "longest_consecutive_anomaly_streak": longest_streak,
    })

summary_df = pd.DataFrame(rows)

# 4) Keep only KPIs with anomalies_count_in_window > 0, sort and limit to top 20
final = (
    summary_df[summary_df["anomalies_count_in_window"] > 0]
    .sort_values(["longest_consecutive_anomaly_streak", "breach_ratio"], ascending=[False, False])
    .reset_index(drop=True)
)

final_top20 = final.head(20)

# Ensure earliest_anomaly_timestamp is integer where present, else NaN (column will be float due to NaNs)
final_top20["earliest_anomaly_timestamp"] = final_top20["earliest_anomaly_timestamp"].apply(lambda x: int(x) if not pd.isna(x) else np.nan)

# Display compact table
final_top20[[
    "kpi_name",
    "global_p95",
    "anomalies_count_in_window",
    "earliest_anomaly_timestamp",
    "max_value_in_window",
    "breach_ratio",
    "longest_consecutive_anomaly_streak"
]]

```
Out[4]:
```
```python
# Summarize checkoutservice KPI anomalies using previously computed `final_top20` DataFrame.
# This prints a compact plain-English summary string (timestamps kept as integer seconds).

try:
    df = final_top20
except NameError:
    # If final_top20 not present, raise to indicate dependency on prior state
    raise

lines = []
lines.append("Checkoutservice metric anomaly summary for window 2024-01-16 00:40:22 to 2024-01-16 01:10:22 UTC")
lines.append("(timestamps shown as integer seconds, global P95 computed from full series)\n")

for _, r in df.iterrows():
    kpi = r["kpi_name"]
    gp95 = r["global_p95"]
    anom = int(r["anomalies_count_in_window"])
    et = int(r["earliest_anomaly_timestamp"]) if not pd.isna(r["earliest_anomaly_timestamp"]) else "NaN"
    mv = r["max_value_in_window"]
    br = r["breach_ratio"]
    streak = int(r["longest_consecutive_anomaly_streak"])
    # format numbers compactly
    def fmt(v):
        if pd.isna(v):
            return "NaN"
        if abs(v) >= 1e6:
            return f"{int(v):,}"  # thousands sep for large numbers
        if abs(v) >= 1:
            return f"{v:.6g}"
        return f"{v:.6g}"
    lines.append(f"- {kpi}: global_p95={fmt(gp95)}, anomalies={anom}, earliest_ts={et}, max_in_window={fmt(mv)}, breach_ratio={fmt(br)}, longest_consecutive_streak={streak}")

lines.append("\nInterpretation: All listed KPIs had 2 points exceeding their global P95 within the window. The largest relative breaches are in latency-90 and latency-50 (breach ratios ~44% and ~20%), so investigate latency first. No KPI shows a multi-minute consecutive anomaly streak (longest streak = 1 for all), indicating isolated spike(s) rather than sustained minute-to-minute failures.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    global_p95  anomalies_count_in_window  earliest_anomaly_timestamp  max_value_in_window  breach_ratio  longest_consecutive_anomaly_streak
0  latency-90  8.816422e-01                          2                  1705365780         1.272944e+00      0.443833                                   1
1  latency-50  3.370723e-01                          2                  1705366500         4.046768e-01      0.200564                                   1
2         cpu  4.656845e-01                          2                  1705366140         5.258337e-01      0.129163                                   1
3    workload  1.024707e+00                          2                  1705366500         1.088917e+00      0.062662                                   1
4         mem  1.221923e+07                          2                  1705366020         1.241675e+07      0.016165                                   1```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present, otherwise load files
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

try:
    df_error_logs
except NameError:
    df_error_logs = pd.read_csv("error_logs.csv")

# Ensure timestamp_dt columns (UTC) exist for clarity (we will keep integer timestamps in outputs)
if "timestamp_dt" not in df_log.columns:
    df_log["timestamp_dt"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)
if "timestamp_dt" not in df_error_logs.columns:
    df_error_logs["timestamp_dt"] = pd.to_datetime(df_error_logs["timestamp"], unit="s", utc=True)

# Incident window (UTC) boundaries as integer seconds
start_ts = int(pd.Timestamp("2024-01-16 00:40:22", tz="UTC").timestamp())
end_ts = int(pd.Timestamp("2024-01-16 01:10:22", tz="UTC").timestamp())

# 1) log.csv: filter checkoutservice within window, group by log_name and aggregate
log_filtered = df_log[
    (df_log["cmdb_id"] == "checkoutservice")
    & (df_log["timestamp"] >= start_ts)
    & (df_log["timestamp"] <= end_ts)
].copy()

log_group = (
    log_filtered
    .groupby("log_name", as_index=False)
    .agg(
        count=("value", "count"),
        min_timestamp=("timestamp", "min"),
        max_timestamp=("timestamp", "max"),
        mean_value=("value", "mean"),
        median_value=("value", "median"),
        max_value=("value", "max"),
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

log_group_top20 = log_group.head(20)

# 2) error_logs.csv: filter checkoutservice within window and return up to first 20 rows with full messages
# Ensure full message display (no truncation) when DataFrame is shown
pd.options.display.max_colwidth = None

error_filtered = df_error_logs[
    (df_error_logs["cmdb_id"] == "checkoutservice")
    & (df_error_logs["timestamp"] >= start_ts)
    & (df_error_logs["timestamp"] <= end_ts)
].copy()

# Select requested columns and keep integer timestamps
error_rows_top20 = error_filtered[["timestamp", "cmdb_id", "message"]].head(20).reset_index(drop=True)

# Return the two compact tables
log_group_top20, error_rows_top20

```
Out[5]:
```
```python
summary = (
    "Checkoutservice logs summary for 2024-01-16 00:40:22 to 2024-01-16 01:10:22 UTC:\n\n"
    "- Aggregated log.csv (grouped by log_name):\n"
    "  • log.error_count: count=25, min_timestamp=1705365780, max_timestamp=1705367220, mean=0.0, median=0.0, max=0\n"
    "  • log.row_count:   count=25, min_timestamp=1705365780, max_timestamp=1705367220, mean=75.72, median=82.0, max=108\n\n"
    "- error_logs.csv: no error log entries for checkoutservice within the incident window.\n\n"
    "Interpretation:\n"
    "- There are no logged error counts or error messages for checkoutservice in the window (log.error_count is 0 and error_logs is empty), so logs do not indicate socket/network errors or explicit application errors for checkoutservice during the incident window.\n"
    "- Previous metric/trace analysis showed latency spikes for checkoutservice; the absence of log errors suggests these were performance spikes (latency) rather than failures logged as errors.\n\n"
    "Recommended next steps:\n"
    "1) Investigate trace latency spikes and downstream calls (currency/payment/email) and their timing.\n"
    "2) Check resource and I/O metrics (CPU, memory, disk, network) for checkoutservice host and related services around the earliest anomaly timestamps.\n"
    "3) If needed, enable more verbose logging around network/socket operations for the narrow time window to capture transient errors.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          log_name  count  min_timestamp  max_timestamp  mean_value  median_value  max_value
0  log.error_count     25     1705365780     1705367220        0.00           0.0          0
1    log.row_count     25     1705365780     1705367220       75.72          82.0        108, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```